# PPR10K Setup — 解压版 (for ECCV rebuttal)

**前提**: 你已经把 PPR10K 的官方 [Drive 文件夹 1kB2OSA...](https://drive.google.com/drive/folders/1kB2OSAGy8uc0xUXaMKoPB0HMSc-rkrLW) 加 shortcut 到 `MyDrive/datasets/`。所以 Colab 看到的路径是:
```
/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p/
    source.zip          12 GB   ← 输入(含 5x 增强,我们只用无增强版)
    target_a.zip         4.6 GB  ← expert a 输出
    target_b.zip         4.5 GB
    target_c.zip         4.7 GB
```

**策略**: 把 source + target_a (~17GB) 解压到 Colab `/content/ppr10k/`,**不要解到 Drive**(Drive 写小文件慢、训练读小文件也慢)。每次 Colab 会话重启后重解一次,大概 3-5 分钟,远比训练时读 Drive 快。

**输出**: 训练数据放 `/content/ppr10k/paired_a/{train,val}/{input,gt}/`,checkpoints 仍存到 Drive 持久化。

In [ ]:
# === Cell: session start — mount Drive + pull latest code ===
# Drive 这份 LoR-LUT 是只读副本,只接收 git pull,从不 commit。
# 跑 cell 时 Colab 会自动把 outputs 写进 .ipynb,git 会看成 'modified'。
# 所以 pull 前先 checkout notebooks/ 丢掉那些自动写入,working tree 才干净。
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/LoR-LUT

# 丢掉 Colab 自动改的 ipynb outputs (永远 OK,我们的 ipynb outputs 不重要)
!git checkout -- notebooks/

# 拉最新代码 (--ff-only: 万一 Drive 那份意外有 commit,立即 abort 不悄悄 merge)
!git pull --ff-only origin main

# 确认拿到最新
!git log --oneline -3

In [ ]:
# === Cell 1: mount + 验证 shortcut 在哪 ===
from google.colab import drive
drive.mount('/content/drive')

import os
PPR_360P = '/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p'
assert os.path.exists(PPR_360P), f"❌ {PPR_360P} 不存在 — 检查 shortcut 是否加在了 MyDrive/datasets/ 下"

print("✅ 找到 PPR10K 360p 目录")
for f in sorted(os.listdir(PPR_360P)):
    full = f'{PPR_360P}/{f}'
    if os.path.isfile(full):
        sz_gb = os.path.getsize(full) / 1e9
        print(f"  📄 {f} ({sz_gb:.2f} GB)")
    else:
        print(f"  📂 {f}/")

In [ ]:
# === Cell 2: 解压 source.zip + target_a.zip 到 /content/ ===
# 经验: source_aug_6 (~57GB,5x augmented data) 在 K=0/R=8 上只贡献 +0.05dB,不值多花 10min 解压 + 25GB /content 空间.
# 默认 USE_AUG=False (只用原始 11161 训练对). 想加增强改成 True.
USE_AUG = False  # ← 改 True 启用 source_aug_6
EXPERT = 'a'
WORK = '/content/ppr10k_raw'

import os, time
os.makedirs(WORK, exist_ok=True)

EXPECTED = {'source': 11161, f'target_{EXPERT}': 11161, 'source_aug_6': 53250}

def maybe_extract(zip_name, parts_dir=None, expected=None):
    """解压. 如果目录已存在但文件数 != expected, 强制重解 (修复磁盘压力下被部分清理的情况)."""
    out_name = zip_name.replace('.zip', '')
    out_dir = f'{WORK}/{out_name}'
    if os.path.exists(out_dir):
        n = len(os.listdir(out_dir))
        if expected and n != expected:
            print(f'⚠️  {out_name} 现有 {n} 文件 != 期望 {expected},清掉重解')
            !rm -rf {out_dir}
        else:
            print(f'⏭️  {out_name} 已解 ({n} 文件)')
            return
    if parts_dir:  # multi-part
        !apt-get -qq install -y p7zip-full > /dev/null
        parts = sorted([f for f in os.listdir(parts_dir) if f.startswith(zip_name + '.')])
        first = f'{parts_dir}/{parts[0]}'
        print(f'📦 解 {zip_name} multi-part ({len(parts)} parts)...')
        t0 = time.time()
        !cd {WORK} && 7z x -bd -y '{first}' > /tmp/7z.log 2>&1 && echo ok || (echo '7z 失败,日志:'; tail -20 /tmp/7z.log)
        print(f'   ✅ {time.time()-t0:.0f}s')
    else:  # single zip
        src_path = f'{PPR_360P}/{zip_name}'
        print(f'📦 解 {zip_name} ({os.path.getsize(src_path)/1e9:.2f} GB)...')
        t0 = time.time()
        !unzip -q -o '{src_path}' -d {WORK}/
        print(f'   ✅ {time.time()-t0:.0f}s')

# 必装: source.zip 和 target_a.zip
maybe_extract('source.zip', expected=EXPECTED['source'])
maybe_extract(f'target_{EXPERT}.zip', expected=EXPECTED[f'target_{EXPERT}'])

# 可选: source_aug_6
if USE_AUG:
    maybe_extract('source_aug_6', parts_dir=f'{PPR_360P}/source_aug_6_zip', expected=EXPECTED['source_aug_6'])
else:
    print('ℹ️  USE_AUG=False, 跳过 source_aug_6')

# 看结构
print('\n=== /content/ppr10k_raw/ 内容 ===')
!ls -la {WORK}/
for d in sorted(os.listdir(WORK)):
    full = f'{WORK}/{d}'
    if os.path.isdir(full):
        files = sorted(os.listdir(full))
        print(f'\n📂 {d}/ ({len(files)} 文件)  头 5: {files[:5]}')

In [ ]:
# === Cell 3: 整理 paired 结构 ===
# DEST 路径根据 USE_AUG 自动选: ppr10k_paired_a_aug / ppr10k_paired_a
import os, re

SOURCE_DIR     = f'{WORK}/source'
TARGET_DIR     = f'{WORK}/target_{EXPERT}'
SOURCE_AUG_DIR = f'{WORK}/source_aug_6' if USE_AUG else None
DEST = f'/content/ppr10k_paired_{EXPERT}_aug' if USE_AUG else f'/content/ppr10k_paired_{EXPERT}'

for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    os.makedirs(f'{DEST}/{sub}', exist_ok=True)

tgt_set = set(os.listdir(TARGET_DIR))

def find_target(source_fname, tgt_set):
    """原始 source: <group>_<photo>.tif -> target 同名;
    增强 source: <group>_<photo>_<aug>.tif -> target 去掉最后 _<aug>."""
    m1 = re.match(r'^(\d+)_(\d+)\.tif$', source_fname)
    if m1 and source_fname in tgt_set:
        return source_fname, (int(m1.group(1)), int(m1.group(2)))
    m2 = re.match(r'^(\d+)_(\d+)_(\d+)\.tif$', source_fname)
    if m2:
        target = f'{m2.group(1)}_{m2.group(2)}.tif'
        if target in tgt_set:
            return target, (int(m2.group(1)), int(m2.group(2)))
    return None, None

# 准备 source list
all_sources = [(SOURCE_DIR, f) for f in sorted(os.listdir(SOURCE_DIR))]
if USE_AUG and os.path.exists(SOURCE_AUG_DIR):
    all_sources += [(SOURCE_AUG_DIR, f) for f in sorted(os.listdir(SOURCE_AUG_DIR))]
print(f'总 source 数 (USE_AUG={USE_AUG}): {len(all_sources)}')

# Split: group_id < 1345 -> train; >= 1345 -> val (1681 groups,~80/20)
TRAIN_GROUP_CUTOFF = 1345
n_train, n_val, n_skip = 0, 0, 0
no_match = []

def link(src, dst):
    if not os.path.exists(dst):
        try: os.symlink(src, dst)
        except FileExistsError: pass

for src_dir, sf in all_sources:
    target_fname, gid_pid = find_target(sf, tgt_set)
    if target_fname is None:
        n_skip += 1
        if len(no_match) < 5: no_match.append(sf)
        continue
    group_id = gid_pid[0]
    split = 'train' if group_id < TRAIN_GROUP_CUTOFF else 'val'
    # val 不用增强 (跟 paper 一致)
    if split == 'val' and src_dir == SOURCE_AUG_DIR:
        continue
    link(f'{src_dir}/{sf}', f'{DEST}/{split}/input/{sf}')
    link(f'{TARGET_DIR}/{target_fname}', f'{DEST}/{split}/gt/{sf}')
    if split == 'train': n_train += 1
    else: n_val += 1

print(f'\n配对结果: train={n_train}, val={n_val}, skipped={n_skip}')
if no_match: print(f'⚠️ 配对失败样本: {no_match}')

print(f'\n✅ DEST = {DEST}')
for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    print(f"   {sub}: {len(os.listdir(f'{DEST}/{sub}'))} 文件")

In [ ]:
# === Cell 4: 验证 paired_folder 能加载,且 input != gt ===
import sys
sys.path.insert(0, '/content/drive/MyDrive/LoR-LUT')
from data.paired_folder import PairedFolderDataset

ds = PairedFolderDataset(
    root=DEST,
    split='train',
    in_dir='input',
    gt_dir='gt',
    exts=('.tif', '.tiff'),
    patch=0,
    augment=False
)
print(f"✅ {len(ds)} train pairs")
for i in [0, 1, 2, len(ds)//2, len(ds)-1]:
    s = ds[i]
    diff = (s['img_in'] - s['img_gt']).abs().mean().item()
    status = '✅' if diff > 0.01 else '⚠️ input==gt!'
    print(f"  [{i:5d}] {s['name']:20s} input-gt MAE={diff:.4f} {status}")

In [ ]:
# === Cell 5: 训练 (用 Cell 3 设的 DEST 路径,自动跟随 USE_AUG) ===
import datetime
tag = '_aug' if USE_AUG else ''
EXP_NAME = f"ppr10k_{EXPERT}_K0_R{__import__('yaml').safe_load(open('/content/drive/MyDrive/LoR-LUT/config/default.yaml'))['model']['R']}{tag}_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
WORK_DIR = f'/content/drive/MyDrive/LoR-LUT/runs/{EXP_NAME}'
print(f'Train data: {DEST}')
print(f'Work dir:   {WORK_DIR}')

%cd /content/drive/MyDrive/LoR-LUT
!python train.py \
    --cfg config/default.yaml \
    --data.root {DEST} \
    --work_dir {WORK_DIR}

## 故障排除

**Cell 2 解压超慢/卡住** —— 检查 Colab 磁盘剩余 (`!df -h /content`)。如果 < 30GB 空间,先 `!rm -rf /content/sample_data` 清出空间。

**Cell 3 配对数 < 100** —— PPR10K 命名跟我猜的不一样。运行下面看实际命名,然后改 Cell 3 的 candidates 列表:
```python
import os
print("source 样本:", sorted(os.listdir(f'{WORK}/source'))[:10])
print("target 样本:", sorted(os.listdir(f'{WORK}/target_a'))[:10])
```

**Cell 4 input==gt** —— 配对错位。Cell 3 里 `link(... target_dir/tf, .../gt/sf)` 这一行的 `tf` (target 文件名) 必须跟 `sf` (source) 是同一张图的不同版本,不是同名两份。

**训练 OOM** —— 改小 batch (`config/default.yaml` 里 `train.batch`),或者 patch (`train.patch`,PPR10K 360p 图本身不大,patch=256 应该够)。